In [6]:
import argparse
import time
import csv
import pickle
import operator
import datetime
import os

dataset = '../../datasets/nowplaying_procesed/raw/nowplaying.csv'

print("-- Starting @ %ss" % datetime.datetime.now())
with open(dataset, "r") as f:
    reader = csv.DictReader(f, delimiter='\t')
    sess_clicks = {}
    sess_times = {}
    sess_date = {}
    ctr = 0
    
    for data in reader:
        sessid = int(data['SessionId'])
        item = int(data['ItemId'])
        timestamp = int(data['Time'])

        if sessid in sess_clicks:
            sess_clicks[sessid].append(item)
            sess_times[sessid].append(timestamp)
        else:
            sess_clicks[sessid] = [item]
            sess_times[sessid] = [timestamp]
            # Use the first recorded interaction as our initial baseline date reference
            sess_date[sessid] = timestamp
        ctr += 1

print('ctr:', ctr)
print("-- Reading data @ %ss" % datetime.datetime.now())

# Filter out length 1 sessions
for s in list(sess_clicks):
    if len(sess_clicks[s]) == 1:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]

# Count number of times each item appears
iid_counts = {}
for s in sess_clicks:
    seq = sess_clicks[s]
    for iid in seq:
        if iid in iid_counts:
            iid_counts[iid] += 1
        else:
            iid_counts[iid] = 1

sorted_counts = sorted(iid_counts.items(), key=operator.itemgetter(1))

# Filter item occurrences and ensure internal chronological order
for s in list(sess_clicks):
    curseq = sess_clicks[s]
    curtime = sess_times[s]
    
    filseq = []
    filtime = []

    # 1. Iterar sobre los índices de la secuencia
    for i in range(len(curseq)):
        item_id = curseq[i]
        
        # 2. Aplicar la condición (El item debe aparecer 5 o más veces)
        if iid_counts[item_id] >= 5:
            filseq.append(item_id)
            filtime.append(curtime[i])

    if len(filseq) < 2 or len(filseq) > 30:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]
    else:
        # --- CHRONOLOGICAL INTERNAL SORT FIX ---
        # Sort items and times together based on the timestamp ascending
        paired = sorted(zip(filtime, filseq), key=lambda x: x[0])
        
        sess_times[s] = [t for t, item in paired]
        sess_clicks[s] = [item for t, item in paired]
        # Dynamically reset the metadata anchor date to the true minimum of sorted values
        sess_date[s] = sess_times[s][0]

# Split out test set based on dates
dates = list(sess_date.items())
maxdate = dates[0][1]

for _, date in dates:
    if maxdate < date:
        maxdate = date

# Two months for test
splitdate = maxdate - 60 * 86400

print('Splitting date', splitdate)
tra_sess = filter(lambda x: x[1] < splitdate, dates)
tes_sess = filter(lambda x: x[1] > splitdate, dates)

# Sort sessions by date
tra_sess = sorted(tra_sess, key=operator.itemgetter(1))
tes_sess = sorted(tes_sess, key=operator.itemgetter(1))
print(len(tra_sess))
print(len(tes_sess))
print(tra_sess[:3])
print(tes_sess[:3])
print("-- Splitting train set and test set @ %ss" % datetime.datetime.now())

item_dict = {}
# Convert training sessions to sequences and renumber items to start from 1
def obtian_tra():
    train_ids = []
    train_seqs = []
    train_times = []
    train_dates = []
    item_ctr = 1
    for s, date in tra_sess:
        seq = sess_clicks[s]
        outseq = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
            else:
                outseq += [item_ctr]
                item_dict[i] = item_ctr
                item_ctr += 1
        if len(outseq) < 2:
            continue
        train_ids += [s]
        train_dates += [date]
        train_seqs += [outseq]
        train_times += [sess_times[s]]
    print('item_ctr')
    print(item_ctr)
    return train_ids, train_dates, train_seqs, train_times


# Convert test sessions to sequences, ignoring items that do not appear in training set
def obtian_tes():
    test_ids = []
    test_seqs = []
    test_dates = []
    test_times = []
    for s, date in tes_sess:
        seq = sess_clicks[s]
        outseq = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
        if len(outseq) < 2:
            continue
        test_ids += [s]
        test_dates += [date]
        test_seqs += [outseq]
        test_times += [sess_times[s]]
    return test_ids, test_dates, test_seqs, test_times


tra_ids, tra_dates, tra_seqs, tra_times = obtian_tra()
tes_ids, tes_dates, tes_seqs, tes_times = obtian_tes()


def process_seqs(iseqs, idates, itimes):
    out_seqs = []
    out_dates = []
    out_times = []
    labs = []
    ids = []
    for id, seq, date, tim in zip(range(len(iseqs)), iseqs, idates, itimes):
        labs.append(seq[-1])
        out_seqs.append(seq[:-1])
        out_dates.append(date)
        out_times.append(tim[:-1])
        ids.append(id)
    return out_seqs, out_dates, out_times, labs, ids


tr_seqs, tr_dates, tr_times, tr_labs, tr_ids = process_seqs(tra_seqs, tra_dates, tra_times)
te_seqs, te_dates, te_times, te_labs, te_ids = process_seqs(tes_seqs, tes_dates, tes_times)

tra = (tr_seqs, tr_labs, tr_times)
tes = (te_seqs, te_labs, te_times)

print('train_test')
print(len(tr_seqs))
print(len(te_seqs))
print(tr_seqs[:3], tr_dates[:3], tr_labs[:3])
print(te_seqs[:3], te_dates[:3], te_labs[:3])
all = 0

for seq in tra_seqs:
    all += len(seq)
for seq in tes_seqs:
    all += len(seq)
print('max_sess_len', max(max([len(s) for s in tr_seqs]), max([len(s) for s in te_seqs])))
print('avg length: ', all/(len(tra_seqs) + len(tes_seqs) * 1.0))
print('all sessions:', all)

NEW_DATASET_FOLDER = "nowplaying_original_sessions"

if not os.path.exists(NEW_DATASET_FOLDER):
    os.makedirs(NEW_DATASET_FOLDER)

def save_as_human_readable(data_list, filename):
    with open(filename, 'w') as f:
        for sequence in data_list:
            if not isinstance(sequence, (list, tuple)) or isinstance(sequence, str):
                sequence = [sequence]
            line = ' '.join(map(str, sequence))
            f.write(line + '\n')
    print(f"Guardado exitosamente como texto legible en: {filename}")


save_as_human_readable(tra, f'{NEW_DATASET_FOLDER}/train_txt.txt')
save_as_human_readable(tes, f'{NEW_DATASET_FOLDER}/test_txt.txt')
# 'seq' wasn't globally persistent across scopes, fixed to write the generated sequences array
save_as_human_readable(tra_seqs, f'{NEW_DATASET_FOLDER}/all_train_seq_txt.txt')

pickle.dump(tra, open(f'{NEW_DATASET_FOLDER}/train.txt', 'wb'))
pickle.dump(tes, open(f'{NEW_DATASET_FOLDER}/test.txt', 'wb'))
pickle.dump(tra_seqs, open(f'{NEW_DATASET_FOLDER}/all_train_seq.txt', 'wb'))

print(f'\nDone. New reduced dataset saved in the "{NEW_DATASET_FOLDER}" folder.')

-- Starting @ 2026-05-23 14:36:31.022067s
ctr: 1587776
-- Reading data @ 2026-05-23 14:36:35.986065s
Splitting date 1429746216
128059
14643
[(231622, 1388710568), (246034, 1388710626), (356572, 1388710633)]
[(222363, 1429746577), (256232, 1429747173), (30241, 1429747483)]
-- Splitting train set and test set @ 2026-05-23 14:36:38.501066s
item_ctr
60417
train_test
128059
14497
[[1, 2, 3, 4, 5], [7], [9, 9]] [1388710568, 1388710626, 1388710633] [6, 8, 10]
[[50393, 701, 12770, 26613, 39979, 6231, 8517, 13481, 4928, 707, 8327], [37136, 24053, 24053, 37136, 2266], [8517, 45787, 57517, 14284, 3772, 6685, 60385, 8517]] [1429746577, 1429747173, 1429747483] [51832, 2266, 38738]
max_sess_len 29
avg length:  7.419428154549791
all sessions: 1057684
Guardado exitosamente como texto legible en: nowplaying_original_sessions/train_txt.txt
Guardado exitosamente como texto legible en: nowplaying_original_sessions/test_txt.txt
Guardado exitosamente como texto legible en: nowplaying_original_sessions/all_t